# SPOTPY calibration steps

**Code under development**
Note: sampler needs set-up and debugging

## Setup

1. Load raven files to Data/Calibration/{model name}


In [ ]:

import os
import sys
import pandas as pd
import spotpy as sp

# Ensure the project directory is in the Python path
prj_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if prj_path not in sys.path:
    sys.path.insert(0, prj_path)
import src.py.raven_py as rv
from src.py.proj_conf import *
from src.py.setup_spotpy import *

## Raven test run

In [ ]:
# Check for Raven.exe in the DATA_CAL folder
raven_exe_path = os.path.join(DATA_CAL, "Raven.exe")
raven_name = "CypressRiverT_ERA5"
# raven_name = input("Enter the Raven model name (if different):")
if os.path.isfile(raven_exe_path):
    test_confirm = input(f"Raven.exe found at {raven_exe_path}. Do you want to run it? (yes or y to confirm): ")
    if (test_confirm.lower() != 'yes' or test_confirm.lower() != 'y'):
        print("Exiting without running Raven.exe.")
    else:
        print(f"Testing Raven.exe at {raven_exe_path}...")
        # Set working directory to DATA_CAL and run Raven.exe
        rv.run_raven(raven_model, DATA_CAL)
else:
    print(f"Raven.exe not found in location: {DATA_CAL}")

## Calibration set-up



In [ ]:
calibrator = spot_setup_rvn(raven_name, DATA_CAL)

calibrator.dep_fact = Uniform(low=0.80, high=1.20)
calibrator.seep_k = Uniform(low=0.00, high=0.10)
calibrator.basin_fact = Uniform(low=0.80, high=1.20)

# sim_results = calibrator.simulation([1,0.05,1])
# obs_results = calibrator.evaluation()


In [ ]:
sampler=sp.algorithms.padds(spot_setup=calibrator, dbname = "padds_raven", dbformat='csv')

In [ ]:
rep = 50
sampler.sample(rep)

In [ ]:
results_df = spotpy.analyser.load_results(sampler.dbname, dbformat=sampler.dbformat)

## Additional workflows

- Extract Raven results

- Specify calibration parameters, method, iterations, etc.
- Run calibration
- Store results & optimal parameters

## Update Raven files

In [ ]:
Cypress_rvp = rv.RavenFile(
    raven_name, calib_dir= DATA_CAL, extension = "rvp")
Cypress_rvp.rvp_update(1,0.05,1)

In [ ]:
Cypress_rvh = rv.RavenFile(
    raven_name, calib_dir= DATA_CAL, extension = "rvh")

Cypress_rvh.rvh_update(1.1,0.05,0.9)